# Import Datasets from Hugging Face

Pull any public Hugging Face dataset into FutureAGI with a single SDK call and run evaluations on it.

By the end of this notebook you will have imported a public Hugging Face dataset into FutureAGI, explored it in the dashboard, run a batch evaluation across every row, and downloaded the scored results.

**Prerequisites:**
- FutureAGI account → [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` ([Get your API keys](https://docs.futureagi.com/admin-settings))
- Python 3.9+

## Install

In [ ]:
%pip install futureagi ai-evaluation --quiet

In [ ]:
import os

os.environ["FI_API_KEY"] = "your-api-key"        # Replace with your key
os.environ["FI_SECRET_KEY"] = "your-secret-key"  # Replace with your key

## Step 1: Import a Hugging Face dataset

Use `HuggingfaceDatasetConfig` to specify which dataset, subset, split, and how many rows to pull. Pass it as the `source` argument to `dataset.create()`.

This example imports 50 rows from the [SmolLM-Corpus](https://huggingface.co/datasets/HuggingFaceTB/smollm-corpus) `cosmopedia-v2` subset — a collection of synthetic textbook-style content with prompts, generated text, audience labels, and format tags.

> **Tip:** `HuggingfaceDatasetConfig` accepts four parameters: `name` (required — the Hugging Face dataset path), `subset` (defaults to `"default"`), `split` (defaults to `"train"`), and `num_rows` (optional — omit to import the entire split).

In [ ]:
import os
from fi.datasets import Dataset, DatasetConfig, HuggingfaceDatasetConfig
from fi.utils.types import ModelTypes

hf_config = HuggingfaceDatasetConfig(
    name="HuggingFaceTB/smollm-corpus",
    subset="cosmopedia-v2",
    split="train",
    num_rows=50,
)

dataset = Dataset(
    dataset_config=DatasetConfig(
        name="smollm-cosmopedia-import",
        model_type=ModelTypes.GENERATIVE_LLM,
    ),
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

dataset = dataset.create(source=hf_config)

print(f"Dataset created: {dataset.dataset_config.name}")
print(f"Dataset ID: {dataset.dataset_config.id}")

## Step 2: View the imported dataset in the dashboard

Navigate to **Dataset** in the left sidebar. Your new dataset appears in the list. Click it to browse the imported rows and columns.

The `cosmopedia-v2` subset includes columns like `prompt`, `text`, `audience`, `format`, and `token_length` — ready for evaluation.

## Step 3: Run an evaluation on the imported data

The `prompt` column contains the generation instruction and `text` contains the generated output — a natural fit for a `completeness` evaluation that checks whether the output fully addresses the input.

> **Note:** Column names depend on the Hugging Face dataset schema. Open the dataset in the dashboard to confirm exact column names before mapping.

In [ ]:
dataset = dataset.add_evaluation(
    name="completeness-check",
    eval_template="completeness",
    required_keys_to_column_names={
        "input": "prompt",
        "output": "text",
    },
    model="turing_small",
    run=True,
    reason_column=True,
)

print("Evaluation 'completeness-check' started")

## Step 4: Download scored results

Pull the evaluated dataset back as a CSV or a pandas DataFrame.

In [ ]:
# As CSV
dataset.download(file_path="smollm_scored.csv")
print("Downloaded scored results to smollm_scored.csv")

In [ ]:
# As pandas DataFrame
df = dataset.download(load_to_pandas=True)
print("Columns:", list(df.columns))
print(df.head())

## Step 5: Clean up

In [ ]:
dataset.delete()
print("Dataset deleted")

## What you built

- Imported 50 rows from the SmolLM-Corpus Hugging Face dataset with a single SDK call
- Browsed the imported data in the FutureAGI dashboard
- Ran a completeness evaluation across every row
- Downloaded scored results as CSV and pandas DataFrame

### Next steps

- [Dataset SDK: Batch Evaluation](https://docs.futureagi.com/cookbook/quickstart/batch-eval) — upload CSVs, add rows, run multi-metric evaluations, and download results
- [Dataset Management](https://docs.futureagi.com/cookbook/quickstart/dataset-management) — create and manage datasets from the dashboard UI
- [Running Your First Eval](https://docs.futureagi.com/cookbook/quickstart/first-eval) — local metrics, Turing models, and LLM-as-Judge with single responses
- [All Built-in Metrics](https://docs.futureagi.com/future-agi/get-started/evaluation/builtin-evals/overview) — full reference for every built-in eval metric